# Tool-calling compliance agent

Inspect the saved real-model run, or set `RUN_MODEL = True` to generate a new one. The controller chooses typed tool calls; the existing pipeline validates the audit and source citations. See `docs/agent-wrapper.md` for the contract and limitations. Expected labels are used only for scoring after inference.


In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / "compliance_copilot").exists():
    ROOT = ROOT.parent
if not (ROOT / "compliance_copilot").exists():
    # Colab setup uses the implementation branch until it is merged.
    ROOT = Path("/content/Aerospace-1C-compliance-copilot")
    if not ROOT.exists():
        subprocess.run([
            "git", "clone", "--branch", "alan/tool-calling-agent",
            "https://github.com/Break-Through-Tech/Aerospace-1C-compliance-copilot.git",
            str(ROOT),
        ], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
sys.path.insert(0, str(ROOT))
print("Project ready:", ROOT.name)


Project ready: Aerospace-1C-compliance-copilot


In [2]:
import json
from compliance_copilot.agent import AgentResult

RUN_MODEL = False
report_path = ROOT / "outputs/agent/demo.json"
if RUN_MODEL:
    report_path = ROOT / "outputs/agent/notebook-run.json"
    subprocess.run([
        sys.executable, str(ROOT / "scripts/run_agent_demo.py"),
        "--output", str(report_path),
    ], check=True)
report = json.loads(report_path.read_text())
runs = [AgentResult.model_validate(run) for run in report["runs"]]
print("Model:", report["config"]["model"])
print("Revision:", report["config"]["model_revision"])


Model: Qwen/Qwen3-4B-Instruct-2507
Revision: cdbee75f17c01a7cc42f958dc650907174af0554


In [3]:
import pandas as pd
from IPython.display import display

display(pd.DataFrame([{
    "requirement": run.result.requirement_id,
    "termination": run.termination,
    "verdict": run.result.audit.verdict if run.result.audit else None,
    "tools": " -> ".join(step.tool or "invalid" for step in run.steps),
    "logged_gaps": len(run.gaps),
} for run in runs]))


,requirement,termination,verdict,tools,logged_gaps
0,NASA-SR-001,finished,Meets,retrieve_clause -> check_requirement -> finish,0
1,NASA-SR-002,finished,Gap,retrieve_clause -> check_requirement -> log_ga...,1
2,NASA-SR-005,finished,Gap,retrieve_clause -> check_requirement -> log_ga...,1


In [4]:
# All source observations and raw controller calls remain available for review.
for run in runs:
    print(run.result.requirement_id)
    print(json.dumps([step.model_dump() for step in run.steps], indent=2))


NASA-SR-001
[
  {
    "step": 1,
    "tool": "retrieve_clause",
    "raw_output": "{\"tool\": \"retrieve_clause\", \"arguments\": {}}",
    "observation": {
      "status": "ok",
      "query": "The NASA Project Manager shall develop, maintain, and execute a Software Development Plan and a Software Security Plan covering the full software life cycle from requirements through retirement, addressing every applicable requirement of NPR 7150.2D per the approved Requirements Mapping Matrix.",
      "clauses": [
        {
          "swe_id": "SWE-013",
          "section": "3.1.3",
          "requirement_text": "3.1.3 The project manager shall develop, maintain, and execute software plans, including security plans, that cover the entire software life cycle and, as a minimum, address the requirements of this directive with approved tailoring. [SWE-013]",
          "source_url": "https://explorers.larc.nasa.gov/APSMEX26/SMEX/pdf_files/NASA21_NPR_7150_2D.pdf#page=20",
          "text_scope": "r

In [5]:
import csv
from scripts.summarize_rag_run import summarize

with (ROOT / "data/benchmark/requirements.csv").open() as stream:
    summary = summarize(report, list(csv.DictReader(stream)))
print(json.dumps({key: value for key, value in summary.items() if key != "details"}, indent=2))
display(pd.DataFrame(summary["details"]))
print("Development examples only. Completed tool calls and valid citations do not prove correct compliance judgments.")


{
  "total": 3,
  "counts": {
    "valid_output": 3,
    "expected_clause_retrieved": 3,
    "expected_clause_cited": 2,
    "verdict_matches": 2,
    "grounded_verdict_matches": 1
  }
}


,requirement_id,expected_verdict,expected_swe_id,predicted_verdict,valid_output,error_code,expected_clause_retrieved,expected_clause_cited,verdict_matches,grounded_verdict_matches
0,NASA-SR-001,Meets,SWE-013,Meets,True,None,True,True,True,True
1,NASA-SR-002,Gap,SWE-034,Gap,True,None,True,False,True,False
2,NASA-SR-005,Partial,SWE-053,Gap,True,None,True,True,False,False


Development examples only. Completed tool calls and valid citations do not prove correct compliance judgments.
